# SpaceX — Dashboard com Plotly Dash (Google Colab)

**Versão simplificada do laboratório da IBM, em português.** Contém as quatro tarefas: seletor da base, gráfico de pizza, seletor de massa e gráfico de dispersão.

### Como usar
1. Clique em **Ambiente de execução → Executar tudo** (ou execute as duas células abaixo, uma após a outra).
2. Aguarde a instalação e o carregamento dos dados.
3. O painel será exibido **abaixo da segunda célula**. Escolha a base e altere a faixa de massa para atualizar os gráficos.
4. Tire capturas de tela do painel para a apresentação da IBM.

**Observação:** o Colab precisa de conexão à internet para baixar o CSV do curso. Não feche a sessão enquanto usar o dashboard.

In [1]:
!pip -q install "dash>=2.11" pandas plotly

In [2]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output


# Arquivo de dados oficial do laboratorio (na mesma pasta do .py).
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
)
DATA_PATH = Path("spacex_launch_dash.csv")


def carregar_dados():
    if DATA_PATH.is_file():
        tabela = pd.read_csv(DATA_PATH)
    else:
        print("CSV nao encontrado na pasta. Tentando baixar o arquivo da IBM...")
        try:
            tabela = pd.read_csv(DATA_URL)
            tabela.to_csv(DATA_PATH, index=False)
            print(f"CSV salvo em: {DATA_PATH}")
        except Exception as exc:
            print("Nao foi possivel baixar o conjunto de dados da IBM.", file=sys.stderr)
            print("Baixe 'spacex_launch_dash.csv' no endereco abaixo e coloque-o", file=sys.stderr)
            print(f"na mesma pasta deste programa: {DATA_PATH.parent}", file=sys.stderr)
            print(DATA_URL, file=sys.stderr)
            raise SystemExit(f"Detalhes do erro: {exc}") from exc

    obrigatorias = [
        "Launch Site", "class", "Payload Mass (kg)", "Booster Version Category"
    ]
    ausentes = [nome for nome in obrigatorias if nome not in tabela.columns]
    if ausentes:
        raise SystemExit(f"O CSV nao possui estas colunas necessarias: {ausentes}")

    # Manter apenas linhas validas para os dois graficos.
    tabela = tabela.copy()
    tabela["class"] = pd.to_numeric(tabela["class"], errors="coerce")
    tabela["Payload Mass (kg)"] = pd.to_numeric(
        tabela["Payload Mass (kg)"], errors="coerce"
    )
    tabela = tabela.dropna(subset=obrigatorias)
    if tabela.empty:
        raise SystemExit("O CSV nao possui registros validos para montar os graficos.")
    tabela["class"] = tabela["class"].astype(int)
    return tabela


spacex_df = carregar_dados()

# Configurar os limites solicitados pela Task 3 (0 a 10.000 kg).
min_payload = max(0, int(spacex_df["Payload Mass (kg)"].min()))
max_payload = min(10000, int(spacex_df["Payload Mass (kg)"].max()))
if max_payload <= min_payload:
    min_payload, max_payload = 0, 10000

# TASK 1: criar a lista suspensa para escolher um local de lancamento.
launch_sites = sorted(spacex_df["Launch Site"].dropna().unique().tolist())
options = [{"label": "Todas as bases", "value": "ALL"}] + [
    {"label": local, "value": local} for local in launch_sites
]

app = Dash(__name__)
app.title = "SpaceX - Dashboard de Lancamentos"

app.layout = html.Div(
    children=[
        html.H1(
            "Dashboard dos lançamentos da SpaceX",
            style={
                "textAlign": "center",
                "color": "#503D36",
                "fontSize": "36px",
                "marginBottom": "22px",
            },
        ),
        html.Label("Selecione a base de lançamento:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="site-dropdown",
            options=options,
            value="ALL",
            placeholder="Selecione uma base de lançamento",
            searchable=True,
            clearable=False,
        ),
        html.Div(
            dcc.Graph(id="success-pie-chart"),
            style={"marginTop": "15px"},
        ),
        html.Label("Intervalo de massa da carga útil (kg):", style={"fontWeight": "bold"}),
        # TASK 3: controle deslizante de massa de 0 a 10.000 kg.
        dcc.RangeSlider(
            id="payload-slider",
            min=0,
            max=10000,
            step=1000,
            marks={i: f"{i:,}".replace(",", ".") for i in range(0, 10001, 2000)},
            value=[min_payload, max_payload],
            allowCross=False,
        ),
        html.Div(
            dcc.Graph(id="success-payload-scatter-chart"),
            style={"marginTop": "25px"},
        ),
        html.P(
            "Dados históricos do exercício da IBM | Class = 1: pouso bem-sucedido; "
            "Class = 0: pouso malsucedido.",
            style={"color": "#666", "textAlign": "center"},
        ),
    ],
    style={"maxWidth": "1100px", "margin": "25px auto", "padding": "0 22px", "fontFamily": "Arial"},
)


# TASK 2: atualizar o grafico de pizza ao selecionar a base de lancamento.
@app.callback(
    Output("success-pie-chart", "figure"),
    Input("site-dropdown", "value"),
)
def atualizar_grafico_pizza(entered_site):
    if entered_site == "ALL":
        # Para todas as bases, comparar apenas o numero de pousos bem-sucedidos.
        data = spacex_df.loc[spacex_df["class"] == 1]
        resumo = data.groupby("Launch Site").size().reset_index(name="Sucessos")
        fig = px.pie(
            resumo,
            values="Sucessos",
            names="Launch Site",
            title="Distribuição dos pousos bem-sucedidos por base",
        )
    else:
        data = spacex_df.loc[spacex_df["Launch Site"] == entered_site]
        resumo = data.groupby("class").size().reset_index(name="Quantidade")
        resumo["Resultado"] = resumo["class"].map({0: "Insucesso", 1: "Sucesso"})
        fig = px.pie(
            resumo,
            values="Quantidade",
            names="Resultado",
            title=f"Pousos bem-sucedidos e malsucedidos: {entered_site}",
            color="Resultado",
            color_discrete_map={"Sucesso": "#2e9d75", "Insucesso": "#d85b5b"},
        )
    fig.update_layout(margin=dict(l=20, r=20, t=65, b=20))
    return fig


# TASK 4: atualizar o grafico de dispersao segundo base e massa selecionadas.
@app.callback(
    Output("success-payload-scatter-chart", "figure"),
    Input("site-dropdown", "value"),
    Input("payload-slider", "value"),
)
def atualizar_grafico_dispersao(entered_site, payload_range):
    if not payload_range or len(payload_range) != 2:
        payload_range = [0, 10000]

    filtrado = spacex_df[
        spacex_df["Payload Mass (kg)"].between(payload_range[0], payload_range[1])
    ]
    if entered_site != "ALL":
        filtrado = filtrado.loc[filtrado["Launch Site"] == entered_site]

    fig = px.scatter(
        filtrado,
        x="Payload Mass (kg)",
        y="class",
        color="Booster Version Category",
        hover_data=["Launch Site"],
        title="Massa da carga útil × resultado do pouso",
        labels={
            "Payload Mass (kg)": "Massa da carga útil (kg)",
            "class": "Resultado do pouso (0 = insucesso; 1 = sucesso)",
            "Booster Version Category": "Versão do foguete",
        },
    )
    fig.update_yaxes(tickvals=[0, 1], range=[-0.2, 1.2])
    fig.update_layout(margin=dict(l=20, r=20, t=65, b=25))
    return fig


# No Colab, o Dash é exibido dentro da última célula, sem abrir terminal.
# Se o painel não aparecer, tente trocar "inline" por "external".
app.run(jupyter_mode="inline", jupyter_height=880, port=8050, debug=False)
